# AOP-ProCon — Step 3 **(build v4)**: Positive-Pair Construction, one complete set of files per experiment mode

Reads the Step-1 **v3 S2** table (1,389 unique sequences, fingerprint `47d9c2a2fff63eb1`) and the Step-2 descriptors, and writes **three complete pair sets** — one per experiment mode — plus one manifest. Step 3 is the only producer of these files.

### Why modes instead of flags (the main change of build v4)

Build v3 wrote one all-peptide pair set and added `strict` / `both_ge4` columns so that Step 4 could filter it. That does **not** give a proper strict or length-restricted experiment: after filtering, the random control is no longer count-matched, some k-NN anchors keep 1–7 of their 8 partners, and the scaler and distance threshold were still fitted on a pool that contained the removed peptides. So the **pool is filtered first** and every step below is repeated on it:

> filter the mechanism pool → fit scaler + p10 threshold on the **fold-0** part of *that* pool → pairs inside each fold → k-NN (K = 8) inside each fold → random control count-matched to *those* pairs → physchem-only control.

| mode | pool | role |
|---|---|---|
| **`strict`** | peptides with `strict_evidence == True` (per-peptide functional or source-reported evidence; no complex/hydrolysate-level or database-only candidates) | **primary analysis** |
| `extended` | all 714 mechanism-active peptides (candidates included, weight 0.5 in training) | sensitivity: does the candidate evidence change the result? |
| `strict_ge4` | strict **and** ≥ 4 aa (`homology_resolvable`) | sensitivity: the length-restricted experiment (3-aa peptides removed from training pairs, validation pairs **and** from the pool that defines prototypes) |

`PRIMARY_MODE = "strict"` is a methodological choice, stated here so that the candidate evidence is a *measured* sensitivity rather than silently part of the main result; change one line if the thesis decision differs.

### Other changes carried over from build v3
S2 fingerprint and descriptor SHA-256 must match `step2_manifest.json`; leak guard on sequence, `cluster_40` **and** `homology_group`; scaler and threshold fitted on the training fold only; pairs and k-NN neighbours found **inside** a fold (no cross-fold pairs, asserted); `d > 0` rule for percentile pairs and k-NN; reproducible random control (fixed seed, asserted); old pair files moved to `data/processed/previous/`, never silently kept.

**What Step 3 does not do:** it does not use `sample_weight` (candidates get weight 0.5 later, in training), and it does not change which sequences exist — that is Step 1's job. It builds **one direction** (train fold 0, validate fold 1); a two-fold cross-validation would need the same construction with the roles reversed.

**Reads / writes** `/content/drive/MyDrive/AOP-ProCon/data/processed/` (pair files in `pairs/<mode>/`)

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

BASE = "/content/drive/MyDrive/AOP-ProCon"

# Resolve BASE instead of trusting it: if the expected location has no AOP-BenchPos_S2.csv, search the whole Drive for it.
if not os.path.exists(f"{BASE}/data/processed/AOP-BenchPos_S2.csv"):
    print(f"NOT at the expected BASE: {BASE}")
    if not os.path.exists("/content/drive/MyDrive"):
        raise RuntimeError("Drive is not mounted -- re-run drive.mount() and complete the authorisation prompt.")
    hits = glob.glob("/content/drive/MyDrive/**/AOP-BenchPos_S2.csv", recursive=True)
    if not hits:
        raise FileNotFoundError("AOP-BenchPos_S2.csv not found under My Drive. Run Steps 1-2 first.")
    BASE = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
    print("resolved BASE ->", BASE)
print("BASE:", BASE)

In [ ]:
import os, json, hashlib, shutil, datetime
from importlib import metadata
import numpy as np, pandas as pd
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler

PROC = f"{BASE}/data/processed"
PREV = f"{PROC}/previous"
os.makedirs(PREV, exist_ok=True)
pd.set_option("display.width", 160)

RNG_SEED = 42
PCT      = 10        # a pair is positive if its descriptor distance is below the p10 of the TRAINING-fold pool
K        = 8         # neighbours per anchor in the k-NN variant
FIT_FOLD = 0         # scaler and threshold are fitted on this fold only (the training fold in Steps 4-5)
MECHS    = ["FRS", "MC"]
MODES    = {"strict": dict(strict=True, ge4=False), "extended": dict(strict=False, ge4=False), "strict_ge4": dict(strict=True, ge4=True)}
PRIMARY_MODE = "strict"      # main analysis; the other modes are sensitivity analyses
PAIR_NAMES = ["positive_pairs", "knn_pairs_k8", "ablation_random_pairs", "ablation_physchem_only_pairs"]
EXPECTED_S2_FINGERPRINT = "47d9c2a2fff63eb1"      # Step 1 build v3

def _ver(pkg):
    try: return metadata.version(pkg)
    except Exception: return "n/a"
VERSIONS = {"python": __import__("sys").version.split()[0], "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": _ver("scipy"), "scikit-learn": _ver("scikit-learn")}
print(VERSIONS)

def s2_fingerprint(s2_df):
    # Canonical content hash of S2 -- identical to Step 1 / Step 2: sequence, labels, homology_group, lso_fold, sample_weight.
    lines = []
    for r in s2_df.sort_values("sequence").itertuples():
        lines.append("\t".join([r.sequence, r.labels,
                                "" if pd.isna(r.homology_group) else str(int(r.homology_group)),
                                "" if pd.isna(r.lso_fold) else str(int(r.lso_fold)),
                                f"{float(r.sample_weight):.1f}"]))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

---
## 1. Load S2 + descriptors and prove they belong together

Three independent checks, all hard assertions:
1. S2 and `descriptors.csv` hold the same sequences, one row each, and are joined **by key** (`sequence`), never by position.
2. The S2 fingerprint equals the one that Step 2 recorded in `step2_manifest.json` (so the descriptors were computed for **this** S2).
3. `descriptors.csv` has the SHA-256 that Step 2 recorded (so it has not been edited since).

In [ ]:
aop  = pd.read_csv(f"{PROC}/AOP-BenchPos_S2.csv")
desc = pd.read_csv(f"{PROC}/descriptors.csv")

required = {"sequence", "labels", "lso_fold", "cluster_40", "homology_group", "strict_evidence", "homology_resolvable", "sample_weight"}
assert required <= set(aop.columns), f"S2 is missing {required - set(aop.columns)} -> not a Step 1 build v3 file; run Step 1 first"
assert aop["sequence"].is_unique,  "S2 must be one row per unique sequence"
assert desc["sequence"].is_unique, "duplicate rows in descriptors.csv"
assert set(aop["sequence"]) == set(desc["sequence"]), "sequence sets differ between S2 and descriptors"

DCOLS = [c for c in desc.columns if c != "sequence"]
assert len(DCOLS) == 14, f"expected 14 descriptors, got {len(DCOLS)}"
assert not desc[DCOLS].isna().any().any(), "NaN in descriptors"

merged = aop.merge(desc, on="sequence", how="inner", validate="one_to_one")
assert len(merged) == len(aop), "merge changed the row count"

S2_FP = s2_fingerprint(aop)
m2 = json.load(open(f"{PROC}/step2_manifest.json"))
assert S2_FP == m2["s2_fingerprint"], f"S2 fingerprint {S2_FP} != Step 2 manifest {m2['s2_fingerprint']}: the descriptors belong to a different S2 -> re-run Step 2"
assert sha256_file(f"{PROC}/descriptors.csv") == m2["sha256"]["descriptors.csv"], "descriptors.csv changed since Step 2 wrote it -> re-run Step 2"
print("S2 fingerprint:", S2_FP, "| Step 2 manifest agrees | descriptors.csv SHA-256 agrees")
if S2_FP != EXPECTED_S2_FINGERPRINT:
    print("NOTE: this is not Step 1 build v3 (", EXPECTED_S2_FINGERPRINT, ") -- consistent with Step 2, but a different split from the one validated in the Step 1 report")

LAB = merged.set_index("sequence")["labels"].to_dict()
print("S2:", aop.shape, "| descriptors:", desc.shape, "| merged:", merged.shape)
print("labels:", merged["labels"].value_counts().to_dict())

### Integrity gate — before anything is written  *(leak guard)*

A pair file built on a leaky split is worthless, so the split is checked first. Sequence, `cluster_40` **and** `homology_group` must each sit in exactly one fold.

In [ ]:
mech_df = merged[merged["labels"].str.contains("FRS|MC")].copy()
assert set(mech_df["labels"].unique()) <= {"FRS", "MC", "FRS;MC"}, "unexpected label value"
assert mech_df["lso_fold"].notna().all() and mech_df["homology_group"].notna().all(), "every mechanism-active sequence needs a fold and a homology group"
assert mech_df.groupby("sequence")["lso_fold"].nunique().max() == 1,       "a sequence is in both folds"
assert mech_df.groupby("cluster_40")["lso_fold"].nunique().max() == 1,     "a cluster_40 id is in both folds"
assert mech_df.groupby("homology_group")["lso_fold"].nunique().max() == 1, "a homology group is in both folds"
assert set(mech_df["lso_fold"].astype(int)) == {0, 1}

FOLD   = mech_df.set_index("sequence")["lso_fold"].astype(int).to_dict()

print("mechanism-active pool:", len(mech_df), "| fold 0:", int((mech_df.lso_fold == 0).sum()), "| fold 1:", int((mech_df.lso_fold == 1).sum()))
print("homology groups:", mech_df["homology_group"].nunique(), "| cluster_40 ids:", mech_df["cluster_40"].nunique())
print("strict evidence:", int(mech_df.strict_evidence.sum()), "| >= 4 aa:", int(mech_df.homology_resolvable.sum()), "| strict and >= 4 aa:", int((mech_df.strict_evidence & mech_df.homology_resolvable).sum()))
print("leak guard: PASS (0 sequences, 0 cluster_40 ids, 0 homology groups span both folds)")

---
## 2. Experiment modes and their mechanism pools — "share ≥ 1 mechanism label"

FRS and MC are **two independent attributes**. A dual (FRS;MC) peptide is a full member of **both** pools; `GEN` peptides are not paired (no mechanism evidence) but are embedded in Step 2 and used as queries in Step 5.

```
FRS pool = { p in mode pool : "FRS" in labels(p) }     MC pool = { p in mode pool : "MC" in labels(p) }     FRS ∩ MC = the dual peptides
```
Each pool is split into its two folds; everything below happens **inside a fold**.

In [ ]:
def mode_pools(mode):
    cfg = MODES[mode]; d = mech_df
    if cfg["strict"]: d = d[d["strict_evidence"].astype(bool)]
    if cfg["ge4"]:    d = d[d["homology_resolvable"].astype(bool)]
    pools = {m: d[d["labels"].str.contains(m)].drop_duplicates("sequence").reset_index(drop=True) for m in MECHS}
    return d.reset_index(drop=True), pools

MODE_POOLS = {mode: mode_pools(mode) for mode in MODES}
rows = []
for mode, (d, pools) in MODE_POOLS.items():
    dual = set(pools["FRS"].sequence) & set(pools["MC"].sequence)
    assert len(dual) > 0, f"{mode}: no peptide is in both pools -> labels are being treated as exclusive"
    for m in MECHS:
        for f in (0, 1):
            rows.append({"mode": mode, "mechanism": m, "fold": f, "n": int((pools[m].lso_fold == f).sum())})
    rows.append({"mode": mode, "mechanism": "dual (both)", "fold": "-", "n": len(dual)})
pool_table = pd.DataFrame(rows)
print(pool_table.pivot_table(index=["mode", "mechanism"], columns="fold", values="n", aggfunc="sum").to_string())
assert MODE_POOLS["extended"][0].shape[0] == len(mech_df)

---
## 3. Build one complete pair set per mode

For each mode and each mechanism:

1. `StandardScaler` fitted on the **fold-0 part of that mode's pool** (14 descriptors).
2. Threshold = 10th percentile of the fold-0 pool's pairwise Euclidean distances.
3. **Percentile pairs**: every pair *inside fold 0* and every pair *inside fold 1* with `0 < d < threshold`.
4. **k-NN pairs (K = 8, directed)**: for every peptide, its 8 nearest admissible neighbours *in its own fold*, same scaler. Self and `d = 0` are set to `inf`, the sort is stable, so the file is identical on every run.
5. **Random control (HP1)**: random same-mechanism same-fold pairs, count-matched per (mechanism, fold) to *this mode's* percentile pairs, fixed seed, asserted reproducible.
6. **Physchem-only control (HP2)**: one mechanism-agnostic pool (`mechanism = "ANY"`), same fold-0 fit and `d > 0` rule. Steps 4–5 do not load this file yet.

**Why `d > 0`:** peptides whose 14 descriptors are identical (for example some anagrams) sit at the same point in descriptor space, so a pair between them carries no distance evidence. The descriptors include order-sensitive features (CTD transitions, instability index), so *not every* anagram pair has distance 0 — the number that do is counted below (`zero_dist_dropped`).

**Why a percentile:** the FRS and MC pools differ several-fold in size; a fixed radius would give MC almost no pairs.

**Why fold 0 for the fit:** fold 0 is the training fold in Steps 4–5. Fold-1 pairs (validation) use the *same* scaler and threshold, so validation is measured on the training-fold geometry.

In [ ]:
def fit_space(P, fit_fold=FIT_FOLD):
    P0 = P[P.lso_fold == fit_fold]
    sc = StandardScaler().fit(P0[DCOLS].values)
    D0 = squareform(pdist(sc.transform(P0[DCOLS].values)))
    return sc, float(np.percentile(D0[np.triu_indices(len(P0), k=1)], PCT))

def threshold_pairs(P, sc, thr, mech):
    X = sc.transform(P[DCOLS].values); D = squareform(pdist(X)); up = np.triu(np.ones_like(D), k=1) > 0
    keep = (D > 0) & (D < thr) & up
    n_zero = int(((D == 0) & up).sum())
    i, j = np.where(keep)
    df = pd.DataFrame({"seq1": P.sequence.values[i], "seq2": P.sequence.values[j], "mechanism": mech, "distance": D[i, j]})
    return df, n_zero, keep.sum(1) + keep.sum(0)

def with_fold(df):
    f1, f2 = df.seq1.map(FOLD), df.seq2.map(FOLD)
    assert (f1 == f2).all(), "a pair spans two folds"
    out = df.copy(); out["fold"] = f1.astype(int)
    return out

def build_mode(mode):
    _, pools = MODE_POOLS[mode]
    res, stats, space = {}, [], {}
    parts = []
    for m in MECHS:
        sc, thr = fit_space(pools[m]); space[m] = (sc, thr)
        for f in (0, 1):
            P = pools[m][pools[m].lso_fold == f].reset_index(drop=True)
            df, n_zero, deg = threshold_pairs(P, sc, thr, m); parts.append(df)
            stats.append({"mode": mode, "mech": m, "fold": f, "n": len(P), "thr(fold0 fit)": round(thr, 3), "pairs": len(df),
                          "zero_dist_dropped": n_zero, "per_anchor": round(float(deg.mean()), 2),
                          "zero_partner": int((deg == 0).sum()), "zero_partner_%": round(float((deg == 0).mean() * 100), 1)})
    res["positive_pairs"] = with_fold(pd.concat(parts, ignore_index=True))

    kparts = []
    for m in MECHS:
        sc, _ = space[m]
        for f in (0, 1):
            P = pools[m][pools[m].lso_fold == f].reset_index(drop=True)
            dist = squareform(pdist(sc.transform(P[DCOLS].values)))
            np.fill_diagonal(dist, np.inf); dist[dist == 0] = np.inf
            assert np.isfinite(dist).sum(1).min() >= K, f"{mode}/{m} fold {f}: an anchor has fewer than {K} admissible neighbours"
            nn = np.argsort(dist, axis=1, kind="stable")[:, :K]
            a = np.repeat(np.arange(len(P)), K); b = nn.ravel()
            kparts.append(pd.DataFrame({"seq1": P.sequence.values[a], "seq2": P.sequence.values[b], "mechanism": m, "distance": dist[a, b]}))
    res["knn_pairs_k8"] = with_fold(pd.concat(kparts, ignore_index=True))
    assert len(res["knn_pairs_k8"]) == sum(len(pools[m]) * K for m in MECHS)
    for m in MECHS:
        for f in (0, 1):
            od = res["knn_pairs_k8"].query("mechanism == @m and fold == @f").seq1.value_counts()
            assert od.min() == od.max() == K, f"{mode}/{m} fold {f}: out-degree is not exactly K"

    def draw_random(seed=RNG_SEED):
        rng = np.random.default_rng(seed); rp = []
        for m in MECHS:
            for f in (0, 1):
                P = pools[m][pools[m].lso_fold == f].reset_index(drop=True)
                n_want = int(((res["positive_pairs"].mechanism == m) & (res["positive_pairs"].fold == f)).sum())
                assert n_want <= len(P) * (len(P) - 1) // 2
                seen = set()
                while len(seen) < n_want:
                    a, b = rng.integers(0, len(P), 2)
                    if a != b: seen.add((int(min(a, b)), int(max(a, b))))
                seen = sorted(seen)
                rp.append(pd.DataFrame({"seq1": [P.sequence.iloc[a] for a, _ in seen], "seq2": [P.sequence.iloc[b] for _, b in seen],
                                        "mechanism": m, "distance": np.nan}))
        return with_fold(pd.concat(rp, ignore_index=True))
    res["ablation_random_pairs"] = draw_random()
    assert res["ablation_random_pairs"].equals(draw_random()), f"{mode}: the random control is not reproducible"
    for m in MECHS:
        for f in (0, 1):
            sel = lambda d: ((d.mechanism == m) & (d.fold == f)).sum()
            assert sel(res["ablation_random_pairs"]) == sel(res["positive_pairs"]), f"{mode}/{m}/fold {f}: random control not count-matched"

    d_mode = MODE_POOLS[mode][0].drop_duplicates("sequence").reset_index(drop=True)
    sc_all, thr_all = fit_space(d_mode)
    pp, zero_all = [], 0
    for f in (0, 1):
        P = d_mode[d_mode.lso_fold == f].reset_index(drop=True)
        df, nz, _ = threshold_pairs(P, sc_all, thr_all, "ANY"); pp.append(df); zero_all += nz
    res["ablation_physchem_only_pairs"] = with_fold(pd.concat(pp, ignore_index=True))
    return res, stats, {m: space[m][1] for m in MECHS} | {"ANY": thr_all}, zero_all

BUILT, ALLSTATS, THRESH = {}, [], {}
for mode in MODES:
    BUILT[mode], st, THRESH[mode], zero_all = build_mode(mode)
    ALLSTATS += st
    print(f"{mode:<11} " + " | ".join(f"{k.replace('_pairs','').replace('ablation_','')}={len(v)}" for k, v in BUILT[mode].items()) + f" | physchem zero-dist dropped={zero_all}")
print()
print(pd.DataFrame(ALLSTATS).to_string(index=False))

---
## 4. Degree distribution, zero-partner tail and dual-peptide exposure (per mode)

Peptides with no partner under the threshold contribute no alignment signal; they are reported, not hidden. A dual peptide should train both prototypes, i.e. appear in FRS pairs **and** in MC pairs, except where it falls in a zero-partner tail.

In [ ]:
for mode, (d_mode, pools) in MODE_POOLS.items():
    pp = BUILT[mode]["positive_pairs"]; dual = set(pools["FRS"].sequence) & set(pools["MC"].sequence)
    print(f"=== {mode} ===")
    for m in MECHS:
        for f in (0, 1):
            d = pp[(pp.mechanism == m) & (pp.fold == f)]
            names = pools[m][pools[m].lso_fold == f].sequence
            deg = pd.concat([d.seq1, d.seq2]).value_counts().reindex(names, fill_value=0)
            print(f"  {m} fold {f}: n={len(names):>3}  degree min={deg.min()} median={deg.median():.0f} max={deg.max()}  zero-partner={(deg == 0).sum()}")
    in_f = set(pp[pp.mechanism == "FRS"].seq1) | set(pp[pp.mechanism == "FRS"].seq2)
    in_m = set(pp[pp.mechanism == "MC"].seq1) | set(pp[pp.mechanism == "MC"].seq2)
    n_both = sum(1 for s in dual if s in in_f and s in in_m)
    assert n_both > 0, f"{mode}: no dual peptide appears in both mechanisms' pairs"
    print(f"  dual peptides in BOTH mechanisms' pairs: {n_both}/{len(dual)}")
    n_frs, n_mc = int((pp.mechanism == 'FRS').sum()), int((pp.mechanism == 'MC').sum())
    print(f"  mechanism imbalance of the percentile pairs: FRS {n_frs} : MC {n_mc} = {n_frs / n_mc:.1f} : 1  (Step 4 must decide how to weight this)")

---
## 4b. How much does the HP1 random control share with the treatment?

`ablation_random` is uniform over the same pool, so some of its pairs fall below the threshold **by chance** and are genuine positive pairs (expected ≈ 10 % under a 10th-percentile rule). The measured overlap is reported so that it can be quoted next to the HP1 result. Whether this overlap changes the *measured model difference* is an empirical question for Steps 4–5, not something to assume.

In [ ]:
def _ukey(df): return {(min(a, b), max(a, b), m) for a, b, m in zip(df.seq1, df.seq2, df.mechanism)}
for mode in MODES:
    pp, rr = BUILT[mode]["positive_pairs"], BUILT[mode]["ablation_random_pairs"]
    ov = _ukey(pp) & _ukey(rr)
    bad = sum(1 for a, b, m in zip(rr.seq1, rr.seq2, rr.mechanism) if m not in LAB[a] or m not in LAB[b])
    assert bad == 0, f"{mode}: random pairs violating their own mechanism label"
    print(f"{mode:<11} overlap of random control with positive pairs: {len(ov)}/{len(rr)} = {100 * len(ov) / len(rr):.2f}%")

---
## 5. Integrity checks, then write

Nothing is written until every assertion passes. Files of an earlier Step-3 build (flat `data/processed/positive_pairs.csv` etc.) and any differing files already in `pairs/<mode>/` are **moved** to `data/processed/previous/`, not deleted. Each mode gets its own folder: `data/processed/pairs/<mode>/`.

In [ ]:
def check(df, name, mode, allow_any=False, directed=False):
    valid = set(MODE_POOLS[mode][0].sequence)
    assert len(df) > 0, f"{mode}/{name} is empty"
    assert list(df.columns) == ["seq1", "seq2", "mechanism", "distance", "fold"], f"{mode}/{name} schema {list(df.columns)}"
    assert df.seq1.isin(valid).all() and df.seq2.isin(valid).all(), f"{mode}/{name}: a pair member is outside this mode's pool"
    assert (df.seq1 != df.seq2).all(), f"{mode}/{name}: self-pair"
    assert (df.seq1.map(FOLD) == df.fold).all() and (df.seq2.map(FOLD) == df.fold).all(), f"{mode}/{name}: pair not inside its fold"
    key = (list(zip(df.seq1, df.seq2, df.mechanism)) if directed
           else [(min(a, b), max(a, b), m) for a, b, m in zip(df.seq1, df.seq2, df.mechanism)])
    assert len(set(key)) == len(key), f"{mode}/{name}: duplicate pair"
    if not allow_any:
        assert set(df.mechanism) <= set(MECHS)
        for m in sorted(set(df.mechanism)):
            s = df[df.mechanism == m]
            assert (s.seq1.map(LAB).str.contains(m) & s.seq2.map(LAB).str.contains(m)).all(), f"{mode}/{name}/{m}: a pair member lacks label {m}"
    assert "FLQL" not in set(df.seq1) | set(df.seq2), f"{mode}/{name}: FLQL (removed from the mechanism tiers) is present"
    if MODES[mode]["strict"]:
        assert df.seq1.map(MODE_STRICT).all() and df.seq2.map(MODE_STRICT).all(), f"{mode}/{name}: a non-strict peptide is present"
    if MODES[mode]["ge4"]:
        assert (df.seq1.str.len() >= 4).all() and (df.seq2.str.len() >= 4).all(), f"{mode}/{name}: a peptide shorter than 4 aa is present"

MODE_STRICT = mech_df.set_index("sequence")["strict_evidence"].astype(bool).to_dict()
print("integrity checks:")
for mode in MODES:
    for name in PAIR_NAMES:
        check(BUILT[mode][name], name, mode, allow_any=(name == "ablation_physchem_only_pairs"), directed=(name == "knn_pairs_k8"))
    print(f"  {mode:<11} OK  " + ", ".join(f"{n.replace('_pairs', '').replace('_k8', '')}={len(BUILT[mode][n])}" for n in PAIR_NAMES))

In [ ]:
def _csv_bytes(df): return df.to_csv(index=False).encode("utf-8")

moved = []
for fn in [f"{n}.csv" for n in PAIR_NAMES] + ["pair_fold_accounting.csv"]:                # flat files of an earlier build
    p = f"{PROC}/{fn}"
    if os.path.exists(p):
        shutil.move(p, f"{PREV}/flat_{fn}"); moved.append(f"flat {fn}")
        print(f"  earlier flat-layout file {fn} moved to previous/flat_{fn}")

for mode in MODES:
    os.makedirs(f"{PROC}/pairs/{mode}", exist_ok=True)
    for name in PAIR_NAMES:
        path = f"{PROC}/pairs/{mode}/{name}.csv"; df = BUILT[mode][name]
        if os.path.exists(path):
            if open(path, "rb").read() == _csv_bytes(df):
                continue
            os.makedirs(f"{PREV}/{mode}", exist_ok=True)
            shutil.move(path, f"{PREV}/{mode}/{name}.csv"); moved.append(f"{mode}/{name}.csv")
            print(f"  {mode}/{name}.csv differed -> old file moved to previous/{mode}/")
        df.to_csv(path, index=False)

acc = []
for mode in MODES:
    for name in PAIR_NAMES:
        d = BUILT[mode][name]
        for m in sorted(set(d.mechanism)):
            for f in (0, 1):
                acc.append({"mode": mode, "set": name, "mechanism": m, "fold": f, "pairs": int(((d.mechanism == m) & (d.fold == f)).sum()), "cross_fold": 0})
pd.DataFrame(acc).to_csv(f"{PROC}/pairs/pair_accounting.csv", index=False)

manifest = {
    "build": "v4 (mode-specific pair construction)", "run_date": str(datetime.date.today()),
    "s2_fingerprint": S2_FP, "s2_rows": int(len(aop)), "descriptors_sha256": sha256_file(f"{PROC}/descriptors.csv"),
    "primary_mode": PRIMARY_MODE, "modes": MODES,
    "parameters": {"percentile": PCT, "k": K, "fit_fold": FIT_FOLD, "seed": RNG_SEED, "d_gt_0_rule": True, "pairs_inside_fold": True,
                   "direction": "train fold 0 / validate fold 1 only"},
    "pool_sizes": {mode: {m: {int(f): int((MODE_POOLS[mode][1][m].lso_fold == f).sum()) for f in (0, 1)} for m in MECHS} for mode in MODES},
    "thresholds_fold0_fit": THRESH,
    "pair_counts": {mode: {n: int(len(BUILT[mode][n])) for n in PAIR_NAMES} for mode in MODES},
    "sha256": {f"{mode}/{n}.csv": sha256_file(f"{PROC}/pairs/{mode}/{n}.csv") for mode in MODES for n in PAIR_NAMES},
    "files_moved_to_previous": moved, "versions": VERSIONS,
}
json.dump(manifest, open(f"{PROC}/pairs_manifest.json", "w"), indent=2)
print("\nWROTE", len(MODES) * len(PAIR_NAMES), "pair files (3 modes x 4 sets) + pairs/pair_accounting.csv + pairs_manifest.json to", PROC)
if moved: print("Models trained on earlier pair files must be retrained.")

### Re-read from disk and confirm  *(a write that was not read back is not a write)*

In [ ]:
mf = json.load(open(f"{PROC}/pairs_manifest.json"))
assert mf["s2_fingerprint"] == S2_FP and mf["primary_mode"] == PRIMARY_MODE
for mode in MODES:
    for n in PAIR_NAMES:
        fn = f"{mode}/{n}.csv"
        assert sha256_file(f"{PROC}/pairs/{fn}") == mf["sha256"][fn], f"{fn}: SHA-256 differs from the manifest"
        d = pd.read_csv(f"{PROC}/pairs/{fn}")
        assert len(d) == mf["pair_counts"][mode][n] and list(d.columns) == ["seq1", "seq2", "mechanism", "distance", "fold"]
        assert (d.seq1.map(FOLD) == d.fold).all() and (d.seq2.map(FOLD) == d.fold).all()
    print(f"{mode:<11} " + " | ".join(f"{n.replace('_pairs', '').replace('_k8', '')}: {mf['pair_counts'][mode][n]}" for n in PAIR_NAMES))
print("\nread-back OK: S2 fingerprint, SHA-256, schema and fold membership agree with the manifest for all 12 files.")
print("Step 4 must LOAD these files by mode; it must not rebuild or overwrite them.")

---
## Summary

Files are written to `data/processed/pairs/<mode>/` for `<mode>` in `strict` (primary), `extended`, `strict_ge4`:

| File | Rule |
|---|---|
| `positive_pairs.csv` | per-mechanism, inside a fold, distance < p10 of the mode's **fold-0** pool |
| `knn_pairs_k8.csv` | per-mechanism, inside a fold, 8 nearest neighbours (directed) |
| `ablation_random_pairs.csv` | random same-mechanism same-fold pairs, count-matched to *this mode's* positive pairs, fixed seed |
| `ablation_physchem_only_pairs.csv` | mechanism-agnostic pool, inside a fold (HP2 control; not yet loaded by Steps 4–5) |

Columns: `seq1, seq2, mechanism, distance, fold`. `pairs/pair_accounting.csv` and `pairs_manifest.json` (S2 fingerprint, parameters, thresholds, counts, SHA-256) record what was written.

### What Step 4 and Step 5 must now do
1. **Retrain every model** — the pair files are new. Primary analysis = `strict`; train the `extended` and `strict_ge4` variants as sensitivity models and compare them on the matching query sets.
2. Assert `pairs_manifest.json["s2_fingerprint"]` and each file's SHA-256 before training or evaluating.
3. **Decide the mechanism weighting explicitly**: FRS pairs outnumber MC pairs by about 23 : 1 (extended), 31 : 1 (strict_ge4) and 57 : 1 (strict) over both folds, and about 40 : 1 in the strict training fold (printed in §4 and in Step 4). A plain average over all pair rows lets FRS dominate the gradient.
4. Remaining limitations: one direction only (train fold 0 / validate fold 1); the pair rule is defined by descriptor similarity, so the supervision signal is only as informative as those 14 descriptors; 3-aa peptides (36 % of mechanism-active sequences) still produce many near-identical pairs that Step 1 could not split across folds (the `strict_ge4` mode removes them); the MC signal is thin — quote the pair counts next to every MC result.